In [ ]:
import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)
import pandas as pd
import healpix as hp
import intake

# Plots
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
import cmocean as cmo
import seaborn as sns
import cartopy.crs as ccrs
import easygems.healpix as egh
###
# Warnings
import warnings
warnings.filterwarnings(action='ignore')

# Global constants

In [ ]:
EARTH_SURFACE = 510e6 # km2
ZOOM_COLORS =  {0: '#5CA0FF', 3: '#006AFF', 6: '#0047AB', 8: '#003175'}
VAR_SHORTNAME = {'2t': r'$\mathrm{T_{2m}}$', 'tp': 'pr', 'sp': r'P$_\mathrm{s}$'}
VAR_UNIT = {'2t': 'K', 'tp': r'mm d$^{-1}$', 'sp': 'Pa'}
VAR_LABEL = {var: VAR_SHORTNAME[var] + ' / ' + VAR_UNIT[var] for var in list(VAR_UNIT.keys())}

# Functions

In [ ]:
def control_freak(factor = 1):
    # Fast function to control plot labels text size
    SMALL_SIZE = int(20 * factor)
    MEDIUM_SIZE = int(22 * factor)
    BIGGER_SIZE = int(26 * factor)

    plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
    plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
    plt.rc('axes', labelsize=SMALL_SIZE)    # fontsize of the x and y labels
    plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
    plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
    plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
    plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title
    return [SMALL_SIZE, MEDIUM_SIZE, BIGGER_SIZE]

In [ ]:
print_basic_info_dataset = lambda ds, var: print(f"From {str(ds.time.values[0])[:10]} to {str(ds.time.values[-1])[:10]} | ncells {ds.cell.size: .2e} |  nside {hp.npix2nside(ds.cell.size)} | " + 
                                                 f"Grid area {EARTH_SURFACE / ds.cell.size :.2f} km2 or spacing of about {np.sqrt(EARTH_SURFACE / ds.cell.size): .2f} km" + 
                                                 f" | nbytes {ds['2t'].nbytes * 1e-6: .2f} MBs" if var is None else f"nbytes {ds[var].nbytes * 1e-6: .2f} MBs")

In [ ]:
f_mean_average_error = lambda ds, ds_ref: np.linalg.norm(ds - ds_ref, 1) / ds.size

In [ ]:
f_intersection = lambda a, b: np.array(list(set(a).intersection(b)))
f_difference = lambda a, b: np.array(list(set(a).difference(b)))

In [ ]:
def get_lon_lat(ds, dim_cell='cell', nest=True):
    nside = hp.npix2nside(ds[dim_cell].size)
    lon, lat = hp.pix2ang(nside, np.arange(ds[dim_cell].size), nest=nest, lonlat=True)
    lon = (lon + 180) % 360 - 180
    return lon, lat

# Data 

As in the previous notebooks, we get the data from the km-scale global hackathon for ERA5 for three different zoom levels, 0, 3, 6, and 8

In [ ]:
cat = intake.open_catalog("https://data.nextgems-h2020.eu/online.yaml")

We want only three variables of those dataset:

- near surface temperature -> `2t` in K
- precipitation -> `tp` measured as total precipitation in meters but we want it in mm d$^{-1}$ (factor = 1e3)
- surface pressure -> `sp`in Pa

For the time dimension, we want to use ERA5 from 2025 to 2020

In [ ]:
variables = ['2t', 'sp', 'tp']

In [ ]:
ds_era5_zoom_0 = cat['ERA5'](zoom=0).to_dask()[variables]
ds_era5_zoom_3 = cat['ERA5'](zoom=3).to_dask()[variables]
ds_era5_zoom_6 = cat['ERA5'](zoom=6).to_dask()[variables]
ds_era5_zoom_8 = cat['ERA5'](zoom=8).to_dask()[variables]
print_basic_info_dataset(ds_era5_zoom_0, None), print_basic_info_dataset(ds_era5_zoom_3, None), print_basic_info_dataset(ds_era5_zoom_6, None), print_basic_info_dataset(ds_era5_zoom_8, None);

Let's only take the relevant years

In [ ]:
ds_era5_zoom_0 = cat['ERA5'](zoom=0).to_dask()[variables].sel(time=slice('1950-01-01', '2020-12-31'))
ds_era5_zoom_3 = cat['ERA5'](zoom=3).to_dask()[variables].sel(time=slice('1950-01-01', '2020-12-31'))
ds_era5_zoom_6 = cat['ERA5'](zoom=6).to_dask()[variables].sel(time=slice('1950-01-01', '2020-12-31'))
ds_era5_zoom_8 = cat['ERA5'](zoom=8).to_dask()[variables].sel(time=slice('1950-01-01', '2020-12-31'))
print_basic_info_dataset(ds_era5_zoom_0, None), print_basic_info_dataset(ds_era5_zoom_3, None), print_basic_info_dataset(ds_era5_zoom_6, None), print_basic_info_dataset(ds_era5_zoom_8, None);

# Work

## Global mean time-series

Since healpix grid data set has equal area cells, we need can just do a mean over the `cell` dimension. 

We are interested in global mean near surface temperature:

In [ ]:
var = '2t'

In [ ]:
%%time
ds_mean_year_zoom_0 = ds_era5_zoom_0[var].mean('cell').compute()

In [ ]:
%%time
ds_mean_year_zoom_3 = ds_era5_zoom_3[var].mean('cell').compute()

In [ ]:
%%time
ds_mean_year_zoom_6 = ds_era5_zoom_6[var].mean('cell').compute()

In [ ]:
%%time
ds_mean_year_zoom_8 = ds_era5_zoom_8[var].mean('cell').compute()

## Some plots

In [ ]:
sns.set_context('talk')
ds_use = [ds_mean_year_zoom_0, ds_mean_year_zoom_3, ds_mean_year_zoom_6, ds_mean_year_zoom_8]
zoom_use = np.array([0, 3, 6, 8])

control_freak(.6)
fig, ax = plt.subplots(1, 4, figsize=(20, 4), constrained_layout=True, facecolor='white', dpi=75)
# Monthly means and yearly mean
for i in range(zoom_use.size):
    ds_use[i].plot(ax=ax[i], color=ZOOM_COLORS[zoom_use[i]], alpha=0.5, zorder=2 + i)
    ds_use[i].resample(time='1Y').mean('time').plot(ax=ax[i], color=ZOOM_COLORS[zoom_use[i]], alpha=1, zorder=3 + i)
    ax[i].set_title(f"Zoom {zoom_use[i]} | MAE with z0: {f_mean_average_error(ds_use[i], ds_use[0]): .3e} K")
[sns.despine(ax=ax_, offset=15) for ax_ in ax]
[ax_.set(xlabel='t / year') for ax_ in ax]
[ax_.set_xlim([pd.Timestamp('1950-01-01'), pd.Timestamp('2021-01-01')]) for ax_ in ax]
[ax_.set(ylabel=VAR_LABEL['2t']) for ax_ in ax]
[ax_.set_ylim([283.5, 290.5]) for ax_ in ax]

fig.tight_layout();
plt.savefig(f"Fig/ERA5_global_mean_{var}_different_zooms.png", format='png', dpi=128, transparent=False, bbox_inches='tight')

## Point-wise operation time-series

Healpix grid is defined by cells, where each cell is a patch over the globe. 
However, we do not know per se from the dataset where does a cell is located in term of latitude and longitude but we can use [hp.ang2pix](https://healpy.readthedocs.io/en/latest/generated/healpy.pixelfunc.ang2pix.html) to obtain the cell that is **closest** to a given coordinate in deg.

Let's explore the impact of resolution to a point-wise analysis. 
To do so, we want to check the precipitation increase in certain location using two dataset with different zoom levels, 3 and 8, between 1950 and 2020.

- Tutunendo in Colombia [-76.54, 5.7]
- Maui Hawaiian island [-156.33, 20.8]
- Ethiopia [40.5, 9.1]

In [ ]:
var = 'tp'

In [ ]:
f_get_point_in_ds = lambda ds, lon, lat: ds.isel(cell=hp.ang2pix(hp.npix2nside(ds.cell.size), lon, lat, nest=True, lonlat=True))

In [ ]:
ds_tutunendo = [f_get_point_in_ds(ds_era5_zoom_3, -76.54, 5.7)[var].load() * 1e3, f_get_point_in_ds(ds_era5_zoom_8, -76.54, 5.7)[var].load() * 1e3]

In [ ]:
ds_maui = [f_get_point_in_ds(ds_era5_zoom_3, -156.33, 20.8)[var].load() * 1e3, f_get_point_in_ds(ds_era5_zoom_8, -156.33, 20.8)[var].load() * 1e3]

In [ ]:
ds_ethiopia = [f_get_point_in_ds(ds_era5_zoom_3, 40.5, 9.1)[var].load() * 1e3, f_get_point_in_ds(ds_era5_zoom_8, 40.5, 9.1)[var'].load() * 1e3]

## Plots

In [ ]:
sns.set_context('talk')
ds_use = ds_maui
zoom_use = np.array([3, 8])

control_freak(.6)
fig, ax = plt.subplots(2, 2, figsize=(20, 4), constrained_layout=True, facecolor='white', dpi=75)
gs = ax[1, 0].get_gridspec()
for ax_ in ax[:, -1]:
    ax_.remove()
axbig = fig.add_subplot(gs[0:, 1])
# Monthly means and yearly mean
for i in range(zoom_use.size):
    ds_use[i].plot(ax=ax[i, 0], color=ZOOM_COLORS[zoom_use[i]], alpha=0.5, zorder=2 + i)
    ds_use[i].resample(time='1Y').mean('time').plot(ax=ax[i, 0], color=ZOOM_COLORS[zoom_use[i]], alpha=1, zorder=3 + i)
    ax[i, 0].set_title(f"Zoom {zoom_use[i]}")
    if i == 0:
        ax[i, 0].set_title(f"Zoom {zoom_use[i]} | MAE with z8: {f_mean_average_error(ds_use[i], ds_use[-1]): .3e} " + VAR_UNIT['tp'])
[sns.despine(ax=ax[i, 0], offset=15) for i in range(2)]
[ax[i, 0].set(xlabel='t / year', ylabel=r'$\mathrm{pr}$ / mm d$^{-1}$') for i in range(2)]
[ax[i, 0].set_xlim([pd.Timestamp('1950-01-01'), pd.Timestamp('2021-01-01')]) for i in range(2)]
#[ax_.set_ylim([283.5, 290.5]) for ax_ in ax]

# Seasonal cycle
for i in range(zoom_use.size):
    mu, std = ds_use[i].groupby("time.month").mean('time'), ds_use[i].groupby("time.month").std('time')
    mu.plot(ax=axbig, color=ZOOM_COLORS[zoom_use[i]], alpha=1, zorder=2 + i)
    axbig.fill_between(mu.month.values, (mu + std).values, (mu - std).values, color=ZOOM_COLORS[zoom_use[i]], alpha=0.25, zorder=1)
axbig.set(xlabel='t / month', ylabel=VAR_LABEL['tp'], title='Seasonal cycle')
axbig.set_xlim([0.5, 12.5])
axbig.set_xticks(np.arange(1, 13))
axbig.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
sns.despine(ax=axbig, offset=15)

fig.tight_layout();

plt.savefig(f"Fig/ERA5_global_mean_and_seasonal_cycle_{var}_z6_vs_z8.png", format='png', dpi=128, transparent=False, bbox_inches='tight')

## Limited-area analysis

Since we can obtain a cell from an specific location, we might wonder if we can do some regional means (limited area analysis).
However, we do not know how many points are in a specific region, thus, [hp.ang2pix](https://healpy.readthedocs.io/en/latest/generated/healpy.pixelfunc.ang2pix.html) might not be enough.

What we can do? We can get an array or mesh with every longitude a latitude that belongs to a certain healpix grid and find all points that are within the region of interest.
There are different way to do it, as creating a mask with True and False for points that satisfy a condtion.
However, operations with mask would require to find always the valid points.
A way to reduce the computational cost would be to get only the cell indexes of interest, thus, we do not need to search the valid points.
Plus, if we have indexes rather than cutting the dataset, it allows backward compatibility of the dataset and no loss of the underlying data structure (grid).

Let's explore the impact of resolution to a limited area analysis. 
To do so, we want to check the precipitation increase in certain location using two dataset with different zoom levels, 3 and 8, between 1950 and 2020.

- The Tibetan Plateau [73 - 105, 26 - 40]
- Costa Rica [-86 - -82.5, 8 - 11.5]

More info in [limited-area analysis](https://easy.gems.dkrz.de/Processing/healpix/limited_area_healpix.html)

In [ ]:
lon_z6, lat_z6 = get_lon_lat(ds_era5_zoom_6)
lon_z8, lat_z8 = get_lon_lat(ds_era5_zoom_8)

In [ ]:
# Tibetean Plateau [73 - 105, 26 - 40]
idx_tibetean_plateau_z6 = f_intersection(np.where((lon_z6 > 73) & (lon_z6 < 105))[0], np.where((lat_z6 > 26) & (lat_z6 < 40))[0])
idx_tibetean_plateau_z8 = f_intersection(np.where((lon_z8 > 73) & (lon_z8 < 105))[0], np.where((lat_z8 > 26) & (lat_z8 < 40))[0])
# Costa Rica [-86 - -82.5, 8 - 11.5]
idx_costa_rica_z6 = f_intersection(np.where((lon_z6 > -86) & (lon_z6 < -82.5))[0], np.where((lat_z6 > 8) & (lat_z6 < 11.5))[0])
idx_costa_rica_z8 = f_intersection(np.where((lon_z8 > -86) & (lon_z8 < -82.5))[0], np.where((lat_z8 > 8) & (lat_z8 < 11.5))[0])

In [ ]:
ds_tibetean_plateau_z6 = ds_era5_zoom_6.sel(cell=idx_tibetean_plateau_z6).load()
ds_tibetean_plateau_z8 = ds_era5_zoom_8.sel(cell=idx_tibetean_plateau_z8).load()

In [ ]:
ds_costa_rica_z6 = ds_era5_zoom_6.sel(cell=idx_costa_rica_z6).load()
ds_costa_rica_z8 = ds_era5_zoom_8.sel(cell=idx_costa_rica_z8).load()

## Plot
Let's see if there are differences in the limited area representation between zoom 3 and zoom 8

In [ ]:
var = 'sp'
region_study = 'Tibetean Plateau' # 'Costa Rica' or 'Tibetean Plateau'
box_region = np.array([73, 105, 26, 40]) if region_study == 'Tibetean Plateau' else np.array([-86, -82.5, 8, 11.5])
bigger_box_region = np.array([60, 120, 15, 55]) if region_study == 'Tibetean Plateau' else np.array([-100, -70, 0, 20])
ds_z6, ds_z8 = [ds_tibetean_plateau_z6[var], ds_tibetean_plateau_z8[var]] if region_study == 'Tibetean Plateau' else [ds_costa_rica_z6[var], ds_costa_rica_z8[var]] 
# Plot
control_freak(1)
cmap = cmo.cm.curl
projection = ccrs.PlateCarree()
fig, ax = plt.subplots(1, 2, figsize=(15, 6), subplot_kw={"projection": projection}, constrained_layout=True, facecolor='white', dpi=60)
# Colobar
bounds = np.linspace(50000, 100500, 21) if region_study == 'Tibetean Plateau' else np.linspace(85000, 101500, 21)
norm = colors.BoundaryNorm(boundaries=bounds, ncolors=256) #colors.Normalize(vmin, vmax)
im = cm.ScalarMappable(norm=norm, cmap=cmap)
cbar = fig.colorbar(im, ax=ax.ravel().tolist(), cax=fig.add_axes([0.1, -0.1, 0.8, 0.05]), orientation='horizontal', extend='both')
cbar.ax.tick_params(labelsize=18)
cbar.set_label(label=VAR_LABEL['sp'], size=17)
#
[ax_.set_global() for ax_ in ax]
[ax_.coastlines() for ax_ in ax]
[ax_.set_extent(bigger_box_region) for ax_ in ax]
[ax_.add_patch(plt.Rectangle(xy=[box_region[0], box_region[2]], width=box_region[1] - box_region[0], height=box_region[3] - box_region[2], facecolor="none", 
                             edgecolor='gray', linewidth=3, transform=ccrs.PlateCarree(), zorder=3)) for ax_ in ax]
for ax_ in ax:
    gl = ax_.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,x_inline=False, y_inline=False, linewidth=2, color='silver', alpha=0.5, linestyle='--')
    gl.xlabel_style = {'size': 15, 'color': 'silver'}
    gl.ylabel_style = {'size': 15, 'color': 'silver'}
    gl.bottom_labels = True
    gl.left_labels   = True
    gl.top_labels    = False
    gl.right_labels  = False
# Zoom 6
ds_use = ds_z6.sel(time=slice('1990-01-01', '1994-12-31')).mean('time').compute()
region = xr.full_like(ds_era5_zoom_6.isel(time=0)['sp'], fill_value=np.nan)
region[np.array(ds_use.cell.values, dtype=int)] = ds_use.values
egh.healpix_show(region, ax=ax[0], cmap=cmap, norm=norm, zorder=2)
ax[0].set(title="Zoom 6")
# Zoom 8
ds_use = ds_z8.sel(time=slice('1990-01-01', '1994-12-31')).mean('time').compute()
region = xr.full_like(ds_era5_zoom_8.isel(time=0)['sp'], fill_value=np.nan)
region[np.array(ds_use.cell.values, dtype=int)] = ds_use.values
egh.healpix_show(region, ax=ax[1], cmap=cmap, norm=norm, zorder=2)
ax[1].set(title="Zoom 8")

fig.suptitle(f"{region_study} " + r"$\overline{\mathrm{1990-1994}}$", x=0.375, y=1.1, 
             fontsize=20, va='top', ha='left', weight='bold');

plt.savefig(f"Fig/ERA5_box_limited_area_{var}_z6_vs_z8.png", format='png', dpi=128, transparent=False, bbox_inches='tight')

# Exercise

## 1 - Limited area time-series analysis 

Do the same time-series analysis performed for point-wise for the limited area for surface pressure and precipitation

- How much error is introduced by using a coarser resolution?
- Is there a difference if you use precipitation instead of surface pressure?

In [ ]:
region_study = 'Tibetean Plateau' # 'Costa Rica' or 'Tibetean Plateau'
var = 'sp'  # ['2t', 'sp', 'tp']
factor = 1e3 if var == 'tp' else 1
# Datasets
ds_z6, ds_z8 = [ds_tibetean_plateau_z6, ds_tibetean_plateau_z8] if region_study == 'Tibetean Plateau' else [ds_costa_rica_z6, ds_costa_rica_z8]
ds_use = [(ds_z6[var] * factor).mean('cell'), (ds_z8[var] * factor).mean('cell')]
zoom_use = [6, 8]

# Plot
sns.set_context('talk')
control_freak(.6)
fig, ax = plt.subplots(2, 2, figsize=(20, 4), constrained_layout=True, facecolor='white', dpi=75)
gs = ax[1, 0].get_gridspec()
for ax_ in ax[:, -1]:
    ax_.remove()
axbig = fig.add_subplot(gs[0:, 1])
# Monthly means and yearly mean
for i in range(2):
    ds_use[i].plot(ax=ax[i, 0], color=ZOOM_COLORS[zoom_use[i]], alpha=0.5, zorder=2 + i)
    ds_use[i].resample(time='1Y').mean('time').plot(ax=ax[i, 0], color=ZOOM_COLORS[zoom_use[i]], alpha=1, zorder=3 + i)
    ax[i, 0].set_title(f"Zoom {zoom_use[i]}")
    if i == 0:
        ax[i, 0].set_title(f"Zoom {zoom_use[i]} | MAE with z8: {f_mean_average_error(ds_use[i], ds_use[-1]): .3e} "  + VAR_UNIT[var])
[sns.despine(ax=ax[i, 0], offset=15) for i in range(2)]
[ax[i, 0].set(xlabel='t / year', ylabel=VAR_LABEL[var]) for i in range(2)]
[ax[i, 0].set_xlim([pd.Timestamp('1950-01-01'), pd.Timestamp('2021-01-01')]) for i in range(2)]
#[ax_.set_ylim([283.5, 290.5]) for ax_ in ax]

# Seasonal cycle
for i in range(2):
    mu, std = ds_use[i].groupby("time.month").mean('time'), ds_use[i].groupby("time.month").std('time')
    mu.plot(ax=axbig, color=ZOOM_COLORS[zoom_use[i]], alpha=1, zorder=2 + i)
    axbig.fill_between(mu.month.values, (mu + std).values, (mu - std).values, color=ZOOM_COLORS[zoom_use[i]], alpha=0.25, zorder=1)
axbig.set(xlabel='t / month', ylabel=VAR_LABEL[var], title='Seasonal cycle')
axbig.set_xlim([0.5, 12.5])
axbig.set_xticks(np.arange(1, 13))
axbig.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
sns.despine(ax=axbig, offset=15)

fig.tight_layout();

plt.savefig(f"Fig/ERA5_box_limited_area_time_series_{var}_z6_vs_z8.png", format='png', dpi=128, transparent=False, bbox_inches='tight')

## 2 - Better ways to do a limited area analysis

From the limited-area analysis we can realize that it looks bad the Tibetean Plateau, as it is not enclosed in a rectangle.
What can we do?

- Create a better way to represent the limited area of the Tibetean plateau.

Hint: python packages: `geopandas` and `shapely` | webpage: [Keene](https://www.keene.edu/campus/maps/tool/)

Bonus tasks:
- Do the same for Costa Rica

In [ ]:
from shapely.geometry import Point, Polygon
import geopandas as gpd

In [ ]:
poly_coords_tibetean_plateau = [(69.2957497, 35.0513595), (71.4492416, 34.1199054), (73.3890152, 34.3543490), (77.1624756, 30.6985970), 
                                (82.6788139, 28.0297128), (87.6729584, 26.7905172), (92.9125786, 26.9480867), (95.5024338, 28.1178649),
                                (96.2416077, 27.5343038), (98.0579567, 27.4372982), (98.2828331, 24.8594937), (100.0000000, 25.0000000),
                                (103.0241203, 25.0283728), (104.5776558, 26.8439837), (103.5318947, 28.5220977), (103.0388832, 29.9698065), 
                                (104.0041351, 31.1462717), (104.6281242, 34.2681406), (103.0376816, 35.5481500), (102.9868698, 37.4743131), 
                                (101.5157318, 38.5473599), (99.5507240, 38.9660826), (96.5540314, 39.8964359), (92.6348305, 40.3831667),
                                (89.1954231, 39.0549178), (84.0598297, 36.9572994), (80.6308937, 36.4124415), (76.8013000, 37.1022889),
                                (74.1125679, 39.5208599), (70.7869720, 38.8589589), (70.8544350, 36.9604543), (69.2957497, 35.0513595)]

In [ ]:
poly_tibetean_plateau = Polygon(poly_coords_tibetean_plateau)

In [ ]:
gdf_z6 = gpd.GeoDataFrame({'Lon': lon_z6, 'Lat': lat_z6}, geometry=gpd.points_from_xy(lon_z6, lat_z6))
gdf_z8 = gpd.GeoDataFrame({'Lon': lon_z8, 'Lat': lat_z8}, geometry=gpd.points_from_xy(lon_z8, lat_z8))

In [ ]:
idx_poly_tibetean_plateau_z6 = gdf_z6[gdf_z6.geometry.within(poly_tibetean_plateau)].index.values
idx_poly_tibetean_plateau_z8 = gdf_z8[gdf_z8.geometry.within(poly_tibetean_plateau)].index.values

In [ ]:
ds_poly_tibetean_plateau_z6 = ds_era5_zoom_6.sel(cell=idx_poly_tibetean_plateau_z6).load()
ds_poly_tibetean_plateau_z8 = ds_era5_zoom_8.sel(cell=idx_poly_tibetean_plateau_z8).load()

In [ ]:
region_study = 'Tibetean Plateau'
box_region = np.array([73, 105, 26, 40])
bigger_box_region = np.array([60, 120, 15, 55])
ds_z6, ds_z8 = [ds_poly_tibetean_plateau_z6['sp'], ds_poly_tibetean_plateau_z8['sp']]
# Plot
control_freak(1)
cmap = cmo.cm.curl
projection = ccrs.PlateCarree()
fig, ax = plt.subplots(1, 2, figsize=(15, 6), subplot_kw={"projection": projection}, constrained_layout=True, facecolor='white', dpi=60)
# Colobar
bounds = np.linspace(50000, 100500, 21) if region_study == 'Tibetean Plateau' else np.linspace(85000, 101500, 21)
norm = colors.BoundaryNorm(boundaries=bounds, ncolors=256) #colors.Normalize(vmin, vmax)
im = cm.ScalarMappable(norm=norm, cmap=cmap)
cbar = fig.colorbar(im, ax=ax.ravel().tolist(), cax=fig.add_axes([0.1, -0.1, 0.8, 0.05]), orientation='horizontal', extend='both')
cbar.ax.tick_params(labelsize=18)
cbar.set_label(label='Surface Pressure / Pa', size=17)
#
[ax_.set_global() for ax_ in ax]
[ax_.coastlines() for ax_ in ax]
[ax_.set_extent(bigger_box_region) for ax_ in ax]
[ax_.add_patch(plt.Rectangle(xy=[box_region[0], box_region[2]], width=box_region[1] - box_region[0], height=box_region[3] - box_region[2], facecolor="none", 
                             edgecolor='gray', linewidth=3, transform=ccrs.PlateCarree(), zorder=3)) for ax_ in ax]
for ax_ in ax:
    gl = ax_.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,x_inline=False, y_inline=False, linewidth=2, color='silver', alpha=0.5, linestyle='--')
    gl.xlabel_style = {'size': 15, 'color': 'silver'}
    gl.ylabel_style = {'size': 15, 'color': 'silver'}
    gl.bottom_labels = True
    gl.left_labels   = True
    gl.top_labels    = False
    gl.right_labels  = False
# Zoom 6
ds_use = ds_z6.sel(time=slice('1990-01-01', '1994-12-31')).mean('time').compute()
region = xr.full_like(ds_era5_zoom_6.isel(time=0)['sp'], fill_value=np.nan)
region[np.array(ds_use.cell.values, dtype=int)] = ds_use.values
egh.healpix_show(region, ax=ax[0], cmap=cmap, norm=norm, zorder=2)
ax[0].set(title="Zoom 6")
# Zoom 8
ds_use = ds_z8.sel(time=slice('1990-01-01', '1994-12-31')).mean('time').compute()
region = xr.full_like(ds_era5_zoom_8.isel(time=0)['sp'], fill_value=np.nan)
region[np.array(ds_use.cell.values, dtype=int)] = ds_use.values
egh.healpix_show(region, ax=ax[1], cmap=cmap, norm=norm, zorder=2)
ax[1].set(title="Zoom 8")

fig.suptitle(f"{region_study} " + r"$\overline{\mathrm{1990-1994}}$", x=0.375, y=1.1, 
             fontsize=20, va='top', ha='left', weight='bold');

plt.savefig(f"Fig/ERA5_poly_limited_area_{var}_z6_vs_z8.png", format='png', dpi=128, transparent=False, bbox_inches='tight')

## 3 - Extra task!

Do the same analysis in 1 with the new dataset and compare the results.

- How much error is introduced by using rough region compared to polygon?
- How much role plays a coarse to a fine resolution by using a polygon?


In [ ]:
region_study = 'Tibetean Plateau'
var = 'sp'  # ['2t', 'sp', 'tp']
factor = 1e3 if var == 'tp' else 1
# Datasets
ds_z6, ds_z8 = [ds_poly_tibetean_plateau_z6, ds_poly_tibetean_plateau_z8]
ds_use = [(ds_z6[var] * factor).mean('cell'), (ds_z8[var] * factor).mean('cell')]
zoom_use = [6, 8]

# Plot
sns.set_context('talk')
control_freak(.6)
fig, ax = plt.subplots(2, 2, figsize=(20, 4), constrained_layout=True, facecolor='white', dpi=75)
gs = ax[1, 0].get_gridspec()
for ax_ in ax[:, -1]:
    ax_.remove()
axbig = fig.add_subplot(gs[0:, 1])
# Monthly means and yearly mean
for i in range(2):
    ds_use[i].plot(ax=ax[i, 0], color=ZOOM_COLORS[zoom_use[i]], alpha=0.5, zorder=2 + i)
    ds_use[i].resample(time='1Y').mean('time').plot(ax=ax[i, 0], color=ZOOM_COLORS[zoom_use[i]], alpha=1, zorder=3 + i)
    ax[i, 0].set_title(f"Zoom {zoom_use[i]}")
    if i == 0:
        ax[i, 0].set_title(f"Zoom {zoom_use[i]} | MAE with z8: {f_mean_average_error(ds_use[i], ds_use[-1]): .3e} "  + VAR_UNIT[var])
[sns.despine(ax=ax[i, 0], offset=15) for i in range(2)]
[ax[i, 0].set(xlabel='t / year', ylabel=VAR_LABEL[var]) for i in range(2)]
[ax[i, 0].set_xlim([pd.Timestamp('1950-01-01'), pd.Timestamp('2021-01-01')]) for i in range(2)]
#[ax_.set_ylim([283.5, 290.5]) for ax_ in ax]

# Seasonal cycle
for i in range(2):
    mu, std = ds_use[i].groupby("time.month").mean('time'), ds_use[i].groupby("time.month").std('time')
    mu.plot(ax=axbig, color=ZOOM_COLORS[zoom_use[i]], alpha=1, zorder=2 + i)
    axbig.fill_between(mu.month.values, (mu + std).values, (mu - std).values, color=ZOOM_COLORS[zoom_use[i]], alpha=0.25, zorder=1)
axbig.set(xlabel='t / month', ylabel=VAR_LABEL[var], title='Seasonal cycle')
axbig.set_xlim([0.5, 12.5])
axbig.set_xticks(np.arange(1, 13))
axbig.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
sns.despine(ax=axbig, offset=15)

fig.tight_layout();

plt.savefig(f"Fig/ERA5_poly_limited_area_time_series_{var}_z6_vs_z8.png", format='png', dpi=128, transparent=False, bbox_inches='tight')

# Questions?